### Importing Dependencies

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document, Prefetch, FusionQuery
from qdrant_client import models
import pandas as pd
import openai
import cohere


/Users/pritam/Desktop/AI-Engineering/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
qdrant_client = QdrantClient(url="http://localhost:6333")


/Users/pritam/Desktop/AI-Engineering/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:290: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


In [3]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=[text],
        model=model,
    )
    return response.data[0].embedding


In [4]:
def retrieve_products(query: str, limit: int = 3):
    """
    Search for products based on a natural language query.
    """
    # 1. Convert the user's text query into a vector embedding
    query_vector = get_embedding(query)
    
    # 2. Search the Qdrant database for the closest matching vectors
    search_results = qdrant_client.query_points(
        collection_name="Products-collection-01-hybrid-search",
        prefetch=[
            Prefetch(
                query=query_vector,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=limit
    )


    retrieve_context = []
    average_rating = []
    similarity_score = []
    context_id = []

    for result in search_results.points:
        # Extract fields from the payload and the Qdrant result object
        retrieve_context.append(result.payload["description"])
        average_rating.append(result.payload["average_rating"])
        similarity_score.append(result.score)
        context_id.append(result.payload["parent_asin"])
        
    return {
        "retrieved_context": retrieve_context,
        "retrieved_context_ratings": average_rating,
        "similarity_score": similarity_score,
        "retrieved_context_ids": context_id
    }

In [6]:
query = "recommend me some shampoos."

In [9]:
results = retrieve_products(query,5)

In [10]:
results

{'retrieved_context': ['Beautiful The Extra Fine Basmati Rice, is rare and not grown anywhere else on earth. This Pusa Basmati (Raw) variety of Kohinoor is the new generation of Basmati. With 7.55 mm long grains, this basmati is selectively picked and aged for around 1-2 years, a reason why even a small quantity of this basmati looks more when cooked and has a little bit of sweetness in its taste with an aroma that will certainly cast a spell on you. is clean hair. There is no reason for a shampoo to contain oils. Oil does not clean hair. Oils and animal byproducts coat the hair with a heavy film leaving hair looking dull and lifeless. Our oilfree extra body shampoo gently cleans the hair. Now hair glows with extra body and more luster. A natural shampoo, recommended for fine, listless hair.',
  'Salon quality hair care proven to moisutrize your hair as well as leading dry remedy shampoos and conditioners',
  'Why Shampoo Bar? Conventional shampoos are harsh on the hair. They use stron

### Reranking

In [12]:
from dotenv import load_dotenv
import os

load_dotenv()  # load .env first

cohere_client = cohere.ClientV2(api_key=os.getenv("CO_API_KEY"))
cohere_client = cohere.ClientV2()


In [13]:
to_rerank = results["retrieved_context"]


In [14]:
to_rerank

['Beautiful The Extra Fine Basmati Rice, is rare and not grown anywhere else on earth. This Pusa Basmati (Raw) variety of Kohinoor is the new generation of Basmati. With 7.55 mm long grains, this basmati is selectively picked and aged for around 1-2 years, a reason why even a small quantity of this basmati looks more when cooked and has a little bit of sweetness in its taste with an aroma that will certainly cast a spell on you. is clean hair. There is no reason for a shampoo to contain oils. Oil does not clean hair. Oils and animal byproducts coat the hair with a heavy film leaving hair looking dull and lifeless. Our oilfree extra body shampoo gently cleans the hair. Now hair glows with extra body and more luster. A natural shampoo, recommended for fine, listless hair.',
 'Salon quality hair care proven to moisutrize your hair as well as leading dry remedy shampoos and conditioners',
 'Why Shampoo Bar? Conventional shampoos are harsh on the hair. They use strong petroleum-based deterg

In [15]:
response = cohere_client.rerank(
    model="rerank-v4.0-pro",
    query=query,
    documents=to_rerank,
    top_n=20
)


In [16]:
response


V2RerankResponse(id='d2e4df09-705a-476d-bf35-eb1dd01c2326', results=[V2RerankResponseResultsItem(index=0, relevance_score=0.82138824), V2RerankResponseResultsItem(index=3, relevance_score=0.8155852), V2RerankResponseResultsItem(index=1, relevance_score=0.7889737), V2RerankResponseResultsItem(index=2, relevance_score=0.62209207), V2RerankResponseResultsItem(index=4, relevance_score=0.28267118)], meta=ApiMeta(api_version=ApiMetaApiVersion(version='2', is_deprecated=None, is_experimental=None), billed_units=ApiMetaBilledUnits(images=None, input_tokens=None, image_tokens=None, output_tokens=None, search_units=1.0, classifications=None), tokens=None, cached_tokens=None, warnings=None))

In [17]:
reranked_results = [to_rerank[result.index] for result in response.results]


In [18]:
reranked_results


['Beautiful The Extra Fine Basmati Rice, is rare and not grown anywhere else on earth. This Pusa Basmati (Raw) variety of Kohinoor is the new generation of Basmati. With 7.55 mm long grains, this basmati is selectively picked and aged for around 1-2 years, a reason why even a small quantity of this basmati looks more when cooked and has a little bit of sweetness in its taste with an aroma that will certainly cast a spell on you. is clean hair. There is no reason for a shampoo to contain oils. Oil does not clean hair. Oils and animal byproducts coat the hair with a heavy film leaving hair looking dull and lifeless. Our oilfree extra body shampoo gently cleans the hair. Now hair glows with extra body and more luster. A natural shampoo, recommended for fine, listless hair.',
 'Sweet Baby Shampoo is a natural, rich, nutritious, well lathering salon quality shampoo for babies and children with no SLS or any sulfates, parabens, phthalates, anesthetizing ingredients, dyes or endocrine disrupt